# T5-bonus · IQ wiring as code

## Goal

Move Fabric workspace/ontology IDs to environment variables with per-
environment `.tfvars`, and assert the Track 5 tenant-setting prerequisites
in code instead of trusting they're still on by the time `prod` deploys.


## Prereqs

Asserted below, not just stated — this cell fails loudly if a prior notebook's step wasn't actually completed.


In [ ]:
from pathlib import Path
assert Path("../agents/contract-renewal-desk/knowledge/fabric-spend-semantic-model.yaml").exists(), "run 16 first"


## Concept

`16`-`19` hardcoded IDs (workspace, ontology) inline for teaching clarity.
Those IDs differ between dev, test, and prod Fabric workspaces — this
notebook moves them to environment variables and per-environment
`.tfvars`, and turns the manual checkpoint prompts from `16`/`18`/`19`
("confirmed? y/n") into an assertion your deploy pipeline can actually run
unattended, using `csx.admin`.


## Build


In [ ]:
import yaml
from pathlib import Path
workspace = Path("../agents/contract-renewal-desk")

for fname, field in [("fabric-spend-semantic-model.yaml", "workspaceId")]:
    path = workspace / "knowledge" / fname
    doc = yaml.safe_load(path.read_text())
    doc[field] = "@{env.fabricWorkspaceId}"
    path.write_text(yaml.dump(doc, sort_keys=False))
print("Fabric workspace ID is now an environment-variable reference")


In [ ]:
tfvars = {
    "dev": {"fabric_workspace_id": "<dev-workspace-guid>"},
    "test": {"fabric_workspace_id": "<test-workspace-guid>"},
    "prod": {"fabric_workspace_id": "<prod-workspace-guid>"},
}
from pathlib import Path
for env_name, values in tfvars.items():
    p = Path(f"../infra/terraform/platform/{env_name}.tfvars")
    p.write_text("\n".join(f'{k} = "{v}"' for k, v in values.items()))
print("per-environment tfvars written")


## Verify

Same harness, same golden set, every notebook.


In [ ]:
from csx.admin import prerequisite_report

# In a real pipeline these come from an actual PPAC/Fabric admin API check,
# not an interactive prompt — this is what T5-bonus replaces the notebook
# checkpoints with.
def check_switches_programmatically():
    # placeholder: wire to real admin API calls before using in a pipeline
    return {"fabric_cross_geo": True}

known = check_switches_programmatically()
report = prerequisite_report(known)
print(report)
assert known.get("fabric_cross_geo") is True, "deploy would fail fast here rather than mid-notebook"


## Cost


In [ ]:
print("No agent build/publish here — variable rebinding and prerequisite checks don't meter.")


## Teardown


In [ ]:
print("No teardown — env-var rebinding is the standing pattern from here through 25.")
